# SBI Stock Price Prediction using RNN — Around Indian Festival Seasons

---

## What This Notebook Does

We train a **Recurrent Neural Network (RNN)** to predict SBI (State Bank of India) stock prices
during windows around major Indian festivals.

**Why festivals?** Stock markets often show unusual volatility around holiday periods —
increased consumer spending, investor sentiment shifts, and low-volume trading days
can create predictable short-term patterns that an RNN can learn.

**Why RNN and not a plain Dense network?**
Stock prices are **sequential** — today's price depends on yesterday's price.
A plain Dense (feedforward) network treats each day independently.
An RNN carries a hidden state forward through time, letting it "remember"
recent price trends when making the next prediction.

---

## Architecture Used

```
Input (scaled price at day t)
        ↓
[SimpleRNN — 50 neurons, ReLU, return_sequences=True]
   ↓ passes full sequence to next RNN layer
[SimpleRNN — 50 neurons, ReLU]
   ↓ final hidden state only
[Dense — 1 neuron, no activation]
        ↓
Output (predicted scaled price at day t+1)
```

---

## Pipeline Overview

```
1. Define Indian festivals with their 2026 dates
2. For each festival: create a ±10-day date window
3. Download SBI stock data (SBIN.NS) for that window via yfinance
4. Normalize prices to [0, 1] using MinMaxScaler
5. Build (X, y) pairs: X = price on day t, y = price on day t+1
6. Train a 2-layer SimpleRNN on 80% of the window data
7. Predict prices for the remaining 20%
8. Inverse-transform predictions back to real rupee values
9. Plot actual vs predicted for every festival window
```

## Step 1 — Import Libraries and Define Festival Dates

Each library has a specific job:
- `pandas`  → tabular data (like a spreadsheet in code)
- `numpy`   → fast numerical arrays and math
- `datetime / timedelta` → date arithmetic (e.g., "10 days before Holi")
- `MinMaxScaler` → normalizes prices from rupees → [0, 1] range
- `Sequential, SimpleRNN, Dense` → building the RNN model
- `matplotlib` → plotting actual vs predicted price charts
- `yfinance` → downloads real stock market data from Yahoo Finance

In [ ]:
# ── STEP 1: IMPORTS ──────────────────────────────────────────────────────────
import pandas as pd                        # tabular data — like an Excel sheet in code
import numpy as np                         # fast math arrays — like double[][] in Java
from datetime import datetime, timedelta   # date arithmetic — add/subtract days

from sklearn.preprocessing import MinMaxScaler
# MinMaxScaler → scales prices from rupees to [0, 1]
# Neural networks train faster and more stably with small, normalized inputs

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, SimpleRNN
# Sequential → stack layers in a straight pipeline
# SimpleRNN  → the recurrent layer that carries a hidden state across time steps
# Dense      → standard fully-connected layer (used at output only here)

import matplotlib.pyplot as plt   # plotting — draw actual vs predicted charts
import yfinance as yf             # Yahoo Finance — downloads real stock price history

print("✅ All libraries imported successfully!")
print()
print("  pandas       → tabular data (DataFrame)")
print("  numpy        → numerical arrays and math")
print("  MinMaxScaler → normalize prices to [0, 1]")
print("  SimpleRNN    → recurrent layer with memory (hidden state)")
print("  Dense        → output layer — predicts a single price value")
print("  yfinance     → free stock data from Yahoo Finance API")

---
## Step 2 — Define Festivals & Build Date Windows

We train a **separate RNN model for each festival**, using only the stock data
from a **±10-day window** around that festival date.

**Why ±10 days?**
- Captures the **pre-festival run-up** (investors buying in anticipation)
- Captures the **post-festival correction** (profit booking after the holiday)
- Keeps each dataset small and focused on one market event

**`get_date_range(festival_date)`** converts a date string like `"2026-03-04"`
into a `(start_date, end_date)` tuple:
```
Holi = 2026-03-04
  start = 2026-02-22  (10 days before)
  end   = 2026-03-14  (10 days after)
```

Run this cell to define all festivals and preview the date windows.

In [ ]:
# ── STEP 2: DEFINE FESTIVALS & BUILD DATE WINDOWS ────────────────────────

# All major Indian festivals in 2026 with their dates
festivals = [
    {"name": "Makar Sankranti",  "date": "2026-01-14"},
    {"name": "Vasant Panchami",  "date": "2026-02-02"},
    {"name": "Maha Shivratri",   "date": "2026-02-17"},
    {"name": "Holi",             "date": "2026-03-04"},
    {"name": "Ugadi",            "date": "2026-03-30"},
    {"name": "Ram Navami",       "date": "2026-04-07"},
    {"name": "Hanuman Jayanti",  "date": "2026-04-13"},
    {"name": "Good Friday",      "date": "2026-04-03"},
    {"name": "Easter",           "date": "2026-04-05"},
    {"name": "Baisakhi",         "date": "2026-04-14"},
    {"name": "Buddha Purnima",   "date": "2026-05-12"},
    {"name": "Eid ul-Adha",      "date": "2026-06-27"},
]

def get_date_range(festival_date):
    """
    Returns (start, end) = ±10 calendar days around the festival.
    Why ±10 days?
      - Pre-festival (days -10 to 0): captures investor anticipation / buying
      - Post-festival (days 0 to +10): captures profit-booking / correction
    """
    dt = datetime.strptime(festival_date, "%Y-%m-%d")
    return dt - timedelta(days=10), dt + timedelta(days=10)

# Build a list of dicts — one entry per festival with computed start/end dates
date_ranges = [
    {
        "festival":      f["name"],
        "festival_date": f["date"],
        "start_date":    get_date_range(f["date"])[0],
        "end_date":      get_date_range(f["date"])[1],
    }
    for f in festivals
]

# Preview all festival windows
print(f"✅ {len(date_ranges)} festival windows created\n")
print(f"{'#':<4} {'Festival':<22} {'Window Start':<15} {'Window End'}")
print("-" * 58)
for i, dr in enumerate(date_ranges):
    print(f"{i:<4} {dr['festival']:<22} {str(dr['start_date'].date()):<15} {dr['end_date'].date()}")

---
## Step 3 — Download Stock Data for One Festival

`yfinance` lets us download real historical stock prices from Yahoo Finance for free.

- **Ticker:** `SBIN.NS` — SBI stock listed on NSE (National Stock Exchange of India)
- **We keep only `Close`** — the price at the end of each trading day
- **Market is closed on weekends and public holidays** — so a 20-calendar-day
  window typically gives around 14 actual trading days

```python
yf.download('SBIN.NS', start=start_date, end=end_date)
# Returns a DataFrame with columns: Open, High, Low, Close, Volume
```

We use the **Holi window** (Feb 22 – Mar 14) as our running demo for Steps 3–9.
Steps 3–9 are **educational walk-throughs** of one festival.
Step 10 runs the complete pipeline for all festivals automatically.

Run this cell to download and preview the Holi window data.

In [ ]:
# ── STEP 3: DOWNLOAD STOCK DATA (demo — Holi window) ─────────────────────
# Using date_ranges[3] = Holi (2026-03-04, window: Feb 22 – Mar 14)
demo = date_ranges[3]
print(f"Downloading SBI data for: {demo['festival']}")
print(f"  Window: {demo['start_date'].date()} → {demo['end_date'].date()}\n")

raw_data = yf.download(
    'SBIN.NS',                       # SBI listed on NSE (National Stock Exchange India)
    start=demo['start_date'].date(),
    end=demo['end_date'].date(),
    progress=False                   # suppress the yfinance progress bar
)

# Keep only the Close column — that is the price at market close each day
data = raw_data[['Close']]

print(f"✅ Downloaded {len(data)} trading days  "
      f"(weekends and market holidays are automatically excluded)\n")
print(data.to_string())   # print all rows so you can inspect the prices

---
## Step 4 — Normalize Prices with MinMaxScaler

Neural networks work best with **small input values** — ideally in [0, 1].
Raw SBI prices (₹700–₹800) would make gradients very large and training unstable.

**MinMaxScaler formula:**
```
scaled = (price - min_price) / (max_price - min_price)
```
- `min_price` → always maps to **0.0**
- `max_price` → always maps to **1.0**
- Every other price falls in between

**Critical rule:** The scaler is fitted once on the training data.
Always reuse the **same scaler** for `inverse_transform` — using a new scaler
fitted on different data would produce wrong ₹ values.

Run this cell to normalize the Holi window prices and see the comparison table.

In [ ]:
# ── STEP 4: NORMALIZE PRICES ─────────────────────────────────────────────
# MinMaxScaler formula:  scaled = (price - min_price) / (max_price - min_price)
#   → min_price maps to 0.0, max_price maps to 1.0, everything else in between
scaler = MinMaxScaler(feature_range=(0, 1))

# fit_transform → learns min/max from data AND applies scaling in one call
scaled_data = scaler.fit_transform(data)   # shape: (n_days, 1)

print(f"Original price range : ₹{data['Close'].min():.2f}  →  ₹{data['Close'].max():.2f}")
print(f"Scaled  value range  :  {scaled_data.min():.4f}  →   {scaled_data.max():.4f}")
print()

# Side-by-side comparison table
comparison = pd.DataFrame({
    'Close (₹)':   data['Close'].values.flatten().round(2),
    'Scaled [0-1]': scaled_data.flatten().round(4),
}, index=data.index.date)
print(comparison.to_string())

---
## Step 5 — Build (X, y) One-Step-Ahead Pairs

The RNN learns to answer: **"Given price on day T, predict price on day T+1."**

We create pairs by shifting the data by one day:
```
X[0] = price on day 0   →   y[0] = price on day 1
X[1] = price on day 1   →   y[1] = price on day 2
...
X[n-2] = price on day n-2  →  y[n-2] = price on day n-1
```

If we have 14 trading days, we get **13 (X, y) pairs**.

**Why `reshape(n, 1, 1)`?**
Keras RNN layers expect input shape `(samples, timesteps, features)`:
- `samples` = number of training examples
- `timesteps = 1` — we feed one day at a time (single-step)
- `features = 1` — one feature (Close price only)

Run this cell to build and inspect the sequence pairs.

In [ ]:
# ── STEP 5: BUILD (X, y) SEQUENCE PAIRS ──────────────────────────────────
# X = all rows except the last   → "price on day T"
# y = all rows except the first  → "price on day T+1"
# Each X[i] is the input; y[i] is the next-day target the model must predict.
X_raw = scaled_data[:-1]   # shape: (n-1, 1)
y_raw = scaled_data[1:]    # shape: (n-1, 1)

# Reshape X to (samples, timesteps=1, features=1) — required by Keras RNN layers
X = X_raw.reshape(X_raw.shape[0], 1, 1)

print(f"X shape : {X.shape}   ← (samples, timesteps=1, features=1)")
print(f"y shape : {y_raw.shape}   ← (samples, 1)\n")

# Preview first 5 (X, y) pairs to verify the one-day offset
print(f"{'Pair':>5}  {'X — day T (scaled)':>20}  {'y — day T+1 (scaled)':>22}")
print("-" * 52)
for i in range(min(5, len(X))):
    print(f"{i:>5}  {X[i][0][0]:>20.4f}  {y_raw[i][0]:>22.4f}")

---
## Step 6 — Train / Test Split (Time-Ordered)

We split **80% train, 20% test** — but crucially **no shuffling**.

Stock prices are sequential: day 10 depends on day 9. If we shuffle and
train on day 12 while testing on day 8, we leak future information into the
past (**data leakage**). Time order must always be preserved.

```
Total: 13 pairs
  Train → first 80% (rows 0–9)
  Test  → last  20% (rows 10–12)
```

Run this cell to see the exact split sizes and array shapes.

In [ ]:
# ── STEP 6: TRAIN / TEST SPLIT ───────────────────────────────────────────
# 80% for training, 20% for testing — NO shuffling (time order must be preserved)
train_size = int(len(X) * 0.8)

X_train = X[:train_size]
X_test  = X[train_size:]
y_train = y_raw[:train_size]
y_test  = y_raw[train_size:]

print(f"Total samples  : {len(X)}")
print(f"Train samples  : {len(X_train)}  (days 0 – {train_size - 1})")
print(f"Test  samples  : {len(X_test)}   (days {train_size} – {len(X) - 1})")
print()
print(f"X_train shape : {X_train.shape}  ← (samples, timesteps, features)")
print(f"X_test  shape : {X_test.shape}")
print(f"y_train shape : {y_train.shape}  ← (samples, 1)")
print(f"y_test  shape : {y_test.shape}")

---
## Step 7 — Build the RNN Model

We stack **two SimpleRNN layers** followed by one Dense output layer.

**Why two RNN layers?**
- Layer 1 learns low-level patterns: "prices rose yesterday"
- Layer 2 learns higher-order patterns: "prices have been rising for 3 days"
- Stacking layers gives the model richer feature extraction

**`return_sequences=True` on Layer 1:**
- By default an RNN only outputs the **last** hidden state
- `return_sequences=True` outputs hidden states for **all** time steps
- Layer 2 needs this — it reads a sequence, not just one vector

**`Dense(1)` output:**
- No activation → direct regression output
- Predicts the next scaled price as a continuous number in [0, 1]

Run this cell to build the model and see its architecture summary.

In [ ]:
# ── STEP 7: BUILD THE RNN MODEL ──────────────────────────────────────────
model = Sequential([

    # Layer 1: reads one timestep at a time, returns ALL hidden states
    #   return_sequences=True → Layer 2 can read each timestep's output
    #   input_shape = (timesteps=1, features=1)
    SimpleRNN(50, activation='relu', return_sequences=True,
              input_shape=(X_train.shape[1], 1)),

    # Layer 2: reads the sequence from Layer 1, returns ONLY final hidden state
    SimpleRNN(50, activation='relu'),

    # Output layer: single neuron, no activation = raw regression value
    Dense(1),
])

model.summary()
print()
print("✅ Model built! Two RNN layers learn patterns at different levels of abstraction.")

---
## Step 8 — Compile and Train the Model

**Compile** attaches the loss function and optimizer to the model.

- **Loss: `mean_squared_error`** — penalizes large errors more than small ones
  - MSE = average of (predicted − actual)²
  - Right choice for regression (predicting a continuous number)

- **Optimizer: `adam`** — Adaptive Moment Estimation
  - Adjusts learning rate automatically per parameter
  - Much better than plain SGD for sequences

**Training loop (50 epochs):**
- Epoch 1: forward pass → compute loss → backprop → update weights
- Epoch 2: repeat with updated weights → loss should decrease
- After 50 epochs: model has learned the price pattern in this window

Run this cell — watch the loss decrease line by line.

In [ ]:
# ── STEP 8: COMPILE AND TRAIN ─────────────────────────────────────────────
model.compile(
    loss='mean_squared_error',   # regression loss: average of (predicted - actual)²
    optimizer='adam'             # adaptive learning rate — best default for RNNs
)

print("Training on Holi festival window…")
print(f"  X_train: {X_train.shape[0]} samples  |  Epochs: 50\n")

history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=20,   # small batch — our dataset is only ~11 training days
    verbose=1        # show loss every epoch so you can watch it decrease
)

print(f"\n✅ Training complete!")
print(f"   Initial loss : {history.history['loss'][0]:.6f}")
print(f"   Final   loss : {history.history['loss'][-1]:.6f}")
print(f"   Loss reduced by {(1 - history.history['loss'][-1]/history.history['loss'][0])*100:.1f}%")

---
## Step 9 — Predict on Test Set & Compare

Now we run the trained model on the **test set** (the last 20% of the window).

The model has **never seen these days** during training — this is a real check
of whether it learned the price pattern or just memorized training data.

**Inverse transform:** The model predicts scaled values in [0, 1].
We convert them back to rupees using `scaler.inverse_transform()`:

```
predicted_scaled  →  scaler.inverse_transform()  →  predicted_price (₹)
```

We also plot actual vs predicted to see how closely they track.

Run this cell to see the comparison table and chart.

In [ ]:
# ── STEP 9: PREDICT ON TEST SET ──────────────────────────────────────────
# Run forward pass on unseen test data (X_test was never used during training)
predicted_scaled = model.predict(X_test, verbose=0)

# Inverse transform both predicted and actual from [0, 1] back to rupees
predicted_prices = scaler.inverse_transform(predicted_scaled.reshape(-1, 1))
actual_prices    = scaler.inverse_transform(y_test.reshape(-1, 1))

# Show comparison table
print(f"{'Day':>5}  {'Actual (₹)':>12}  {'Predicted (₹)':>15}  {'Error (₹)':>12}")
print("-" * 52)
for i in range(len(actual_prices)):
    err = predicted_prices[i][0] - actual_prices[i][0]
    print(f"{i+1:>5}  {actual_prices[i][0]:>12.2f}  {predicted_prices[i][0]:>15.2f}  {err:>+12.2f}")

# Quick plot: actual vs predicted
plt.figure(figsize=(8, 4))
plt.plot(actual_prices,    label='Actual',    marker='o', linewidth=2)
plt.plot(predicted_prices, label='Predicted', marker='s', linewidth=2, linestyle='--')
plt.title('Holi Window — Actual vs Predicted SBI Price')
plt.xlabel('Test Day')
plt.ylabel('Price (₹)')
plt.legend()
plt.tight_layout()
plt.show()

print("\n✅ Note: Model trained on only a few days — educational only, not financial advice.")

---
## Step 10 — Full Pipeline: All Festival Windows

Steps 3–9 walked through the pipeline for **one festival (Holi)** step by step.

Now we run the **complete pipeline for ALL festivals** in one go:
1. Download data for each festival window
2. Normalize, create sequences, split 80/20
3. Train a **fresh RNN model** per festival window
4. Predict and plot actual vs predicted for each

Each festival gets its own model because each window has different price
levels and volatility patterns — one shared model would not capture this.

Run this cell — it produces one chart per festival.

In [ ]:
# ── STEP 10: FULL PIPELINE — ALL FESTIVALS ───────────────────────────────

def fetch_and_train_model(date_range):
    """Train an RNN on one festival window and return predictions + trained objects."""
    print(f"  Processing: {date_range['festival']}  "
          f"({date_range['start_date'].date()} → {date_range['end_date'].date()})")
    try:
        # Download
        data = yf.download('SBIN.NS',
                           start=date_range['start_date'].date(),
                           end=date_range['end_date'].date(),
                           progress=False)
        data = data[['Close']]

        # Normalize
        scaler_f = MinMaxScaler(feature_range=(0, 1))
        scaled   = scaler_f.fit_transform(data)

        # Sequences
        X_f = scaled[:-1].reshape(-1, 1, 1)
        y_f = scaled[1:]

        # Split 80/20
        split      = int(len(X_f) * 0.8)
        X_tr, X_te = X_f[:split], X_f[split:]
        y_tr, y_te = y_f[:split], y_f[split:]

        # Build model
        m = Sequential([
            SimpleRNN(50, activation='relu', return_sequences=True, input_shape=(1, 1)),
            SimpleRNN(50, activation='relu'),
            Dense(1),
        ])
        m.compile(loss='mean_squared_error', optimizer='adam')
        m.fit(X_tr, y_tr, epochs=50, batch_size=20, verbose=0)

        # Predict (inverse-transform to ₹)
        pred = scaler_f.inverse_transform(m.predict(X_te, verbose=0).reshape(-1, 1))
        act  = scaler_f.inverse_transform(y_te.reshape(-1, 1))

        return data.index[-len(y_te):], act, pred, m, scaler_f, data

    except Exception as e:
        print(f"    ⚠ Error: {e}")
        return [], [], [], None, None, None


def plot_all_festivals(date_ranges):
    n    = len(date_ranges)
    rows = (n + 1) // 2
    fig, axes = plt.subplots(nrows=rows, ncols=2, figsize=(14, 4 * rows))
    axes = axes.flatten()

    results = {}
    for i, dr in enumerate(date_ranges):
        dates, actual, predicted, m, s, d = fetch_and_train_model(dr)
        results[dr['festival']] = (m, s, d)

        if len(actual) > 0:
            axes[i].plot(actual,    label='Actual',    marker='o', linewidth=1.5)
            axes[i].plot(predicted, label='Predicted', marker='s', linewidth=1.5, linestyle='--')
            axes[i].set_title(dr['festival'], fontsize=11)
            axes[i].set_xlabel('Test Day')
            axes[i].set_ylabel('Price (₹)')
            axes[i].legend(fontsize=8)
        else:
            axes[i].set_title(f"{dr['festival']} — no data", fontsize=10)

    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])

    plt.suptitle('SBI Stock Price — RNN Predictions Around Indian Festivals',
                 fontsize=13, y=1.01)
    plt.tight_layout()
    plt.show()
    return results


print("Running full pipeline for all festivals…")
all_results = plot_all_festivals(date_ranges)
print("\n✅ All festivals processed! Check the charts above.")

---
## Step 11 — Predict SBI Price for a Specific Future Date

This step uses the **trained model** from one festival window to predict the price
on a specific future date within that window.

**How it works:**
1. Pick a target date (e.g., `"2026-05-02"`)
2. Select the festival window that **contains** that date
3. Train the model on that window
4. Find the last known price **before** the target date
5. Scale → feed to model → inverse-scale → print ₹ value

**Your task:** Predict SBI price for **2nd May 2026** and record your answer.
We'll check accuracy once the market opens on 2nd May!

In [ ]:
# ── STEP 11: PREDICT FOR A SPECIFIC DATE ─────────────────────────────────

predict_for_date  = "2026-05-02"    # ← change to any date inside a festival window

# Buddha Purnima window = date_ranges[10]  (May 02 – May 22 approx)
target_date_range = date_ranges[10]
print(f"Selected window: {target_date_range['festival']}")
print(f"  {target_date_range['start_date'].date()} → {target_date_range['end_date'].date()}\n")

# ① Train the model on this window (re-uses fetch_and_train_model from Step 10)
dates, actual_p, predicted_p, model_s11, scaler_s11, data_s11 = \
    fetch_and_train_model(target_date_range)

# ② Find the last KNOWN trading day before the target date
target_dt       = pd.to_datetime(predict_for_date)
available_dates = data_s11.index[data_s11.index < target_dt]
last_date       = available_dates[-1]

# .iloc[0] safely extracts the scalar float from the DataFrame cell
last_known_price = float(data_s11.loc[last_date, "Close"].iloc[0])
print(f"Last known price ({last_date.date()}): ₹{last_known_price:.2f}")

# ③ Scale using the SAME scaler (critical — never create a new scaler here)
#    scaler.transform([[value]]) → [[scaled_value]]
#    reshape(1,1,1) → (samples=1, timesteps=1, features=1) for RNN input
last_scaled = scaler_s11.transform([[last_known_price]]).reshape(1, 1, 1)

# ④ Forward pass through the trained model → scaled prediction in [0, 1]
predicted_scaled = model_s11.predict(last_scaled, verbose=0)

# ⑤ Inverse transform: scaled = (value - min) / (max - min)
#    → value = scaled × (max - min) + min
predicted_price = scaler_s11.inverse_transform(predicted_scaled)[0][0]

print(f"\nPredicted SBI price for {predict_for_date}: ₹{predicted_price:.2f}")
print()
print("⚠  Model trains on only ~14 trading days — educational use only, not financial advice.")

## **Your Task to predict the SBI price for 2nd May, We will wait and watch your prediction accuracy on 2nd May**